# Phase 5A Lab - Router And Synthesizer Deterministic

Mục tiêu: hiểu assistant behavior deterministic: Router tiếng Việt,
Synthesizer tiếng Việt, progress steps, summary cards.

Expected output chính: Router/Synthesizer tests pass; direct demos trả
`search_deals`, `query_en`, Vietnamese answer và sorted cards.

Safety: không OpenAI Agents SDK, không model calls.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def find_v3_root(start: str | None = None) -> Path:
    path = Path(start or os.getcwd()).resolve()
    for candidate in (path, *path.parents):
        if candidate.name == "shopping_assistant_v3" and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the tech2ai/shopping_assistant_v3 tree.")


V3_ROOT = find_v3_root()
os.chdir(V3_ROOT)
if str(V3_ROOT) not in sys.path:
    sys.path.insert(0, str(V3_ROOT))


def run(command: list[str], timeout: int = 120) -> subprocess.CompletedProcess[str] | None:
    print("$ " + " ".join(command))
    try:
        result = subprocess.run(
            command,
            cwd=V3_ROOT,
            text=True,
            capture_output=True,
            timeout=timeout,
        )
    except subprocess.TimeoutExpired:
        print(f"Command timed out after {timeout} seconds.")
        return None

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print(f"exit_code={result.returncode}")
    return result


print(f"V3_ROOT={V3_ROOT}")
print("Default safety flags:")
for name in ("ENABLE_REAL_SEARCH", "ENABLE_REAL_MODEL_CALLS", "ENABLE_AGENTS_SDK"):
    print(f"{name}={os.getenv(name, '<unset>')}")


## 1. Chạy focused Router/Synthesizer tests


In [ ]:
run(["uv", "run", "pytest", "tests/test_router.py", "tests/test_synthesizer.py", "-q", "--tb=short"], timeout=180)


## 2. Demo deterministic Router

Expected: shopping query trả intent `search_deals`, source `All`, và English
query.


In [ ]:
from backend.router.deterministic import deterministic_route

for message in [
    "Tìm laptop gaming dưới 800 đô",
    "Tìm tai nghe trên Amazon",
    "Kể chuyện cười đi",
]:
    route = deterministic_route(message)
    print(message)
    print(route.model_dump())
    print()


## 3. Demo Synthesizer từ evidence

Expected: output tiếng Việt, prices giữ USD, cards sort theo discount.


In [ ]:
from backend.synthesizer.deterministic import deterministic_synthesize
from backend.synthesizer.schemas import SynthesizerInput
from backend.tools.deal_search.schemas import ProductCandidate
from backend.tools.price_estimator.schemas import ModelBreakdown, PriceEstimateOutput

products = [
    ProductCandidate(source="Amazon", title="Example Laptop A", brand="Example", sale_price_usd=699.99, url="https://www.amazon.com/a", features="16GB RAM"),
    ProductCandidate(source="BestBuy", title="Example Laptop B", brand="Example", sale_price_usd=799.99, url="https://www.bestbuy.com/b", features="RTX GPU"),
]
estimates = [
    PriceEstimateOutput(estimated_value_usd=820.0, discount_usd=120.01, deal_score="good", confidence=None, model_breakdown=ModelBreakdown(frontier=820.0, specialist=820.0, neural=820.0), warnings=[]),
    PriceEstimateOutput(estimated_value_usd=1050.0, discount_usd=250.01, deal_score="hot", confidence=None, model_breakdown=ModelBreakdown(frontier=1050.0, specialist=1050.0, neural=1050.0), warnings=[]),
]
synth = deterministic_synthesize(SynthesizerInput(
    message_vi="Tìm laptop gaming dưới 800 đô",
    intent="search_deals",
    products=products,
    price_estimates=estimates,
    warnings=[],
))
print(synth.answer_vi)
print(json.dumps([card.model_dump() for card in synth.summary_cards], indent=2, ensure_ascii=False))


## 4. Demo progress steps

Expected: đúng 4 fixed steps, không phải LLM-generated todo list.


In [ ]:
from backend.shared.progress import build_progress_steps

steps = build_progress_steps(
    route_completed=True,
    search_completed=True,
    pricing_completed=True,
    synth_completed=True,
)
for step in steps:
    print(step.model_dump())


## 5. Cách đọc kết quả

- Router unsupported path có `needs_tool=False`.
- Synthesizer không bịa URL/price ngoài products/estimates truyền vào.
- `summary_cards` sort theo discount.
- Progress luôn có 4 step ids: `route_request`, `search_deals`,
  `estimate_prices`, `synthesize_answer`.
